# E2 — blinded identity audit of the deterministic gate

**SESAME claim this notebook serves:** **S2** (`docs/CURRENT.md`).
**Plans:** `docs/plan/2026-08-31-e2-identity-audit-codebook.md`,
`docs/plan/2026-09-01-e1-structured-check-validation.md`.
**Not this notebook:** gate traceability accounting (E1), evaluator dependence (E3).

E1 showed the gate's decisions are *inspectable*. This notebook asks the different
question of whether they are *right*: for each unique
`(entity_type, lowercased surface, candidate_uri)` decision, a human who could not see
the machine outcome judged the candidate URI itself.

Because reviewers were blinded, the label is a verdict on the **candidate**, not on the
gate. Unblinding maps it onto the gate as follows (codebook §"After unblinding"):

| Gate outcome | Human label | Gate was |
|---|---|---|
| accept | `correct` | right — true accept |
| accept | `wrong` or `true_nil` | **wrong — false accept** |
| refuse | `wrong` or `true_nil` | right — correct refusal |
| refuse | `correct` | wrong — false refusal |

`indeterminate` is excluded from every rate and reported separately; it is not evidence
either way.

Run **top to bottom** after a fresh kernel.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

In [2]:
# ── CONFIG (edit this cell; then Run All) ──────────────────────────────
ANALYSIS_DIR = Path(
    r"<path>"
)
EXPERIMENTS_DIR = ANALYSIS_DIR.parent
FREEZE = EXPERIMENTS_DIR / "data" / "freeze" / "e2_identity_labels.jsonl"
ARTIFACT = EXPERIMENTS_DIR / "artifacts" / "e2_identity_audit_summary.json"
OUTPUT_DIR = ANALYSIS_DIR / "output"

# True  = deliberately replace the submission freeze from the live review database.
# False = analyse the 2026-09-05 21:30 UTC submission freeze exactly as it stands.
# Review may continue in the portal, but later labels are outside this analysis cutoff.
REFRESH_FROM_DB = False

# Wilson score-interval coverage. 1.96 = 95%.
Z = 1.96

# A pause longer than this is a break, not review time (used for wall-clock effort).
BREAK_THRESHOLD_S = 300

# Reviewer -> assigned sort_order block. Mirrors review_app/identity.py.
BLOCKS = {
    "R1": (1, 100), "R2": (101, 200), "R3": (201, 300),
    "R4": (301, 400), "R5": (401, 500),
}

# On an accepted row these both mean the gate linked the wrong thing; only the
# reviewer's explanation differs. Section 4 shows why keeping them merged matters.
GATE_ERROR_LABELS = ("wrong", "true_nil")

LABEL_ORDER = ["correct", "wrong", "true_nil", "indeterminate"]
OUTCOME_LABEL = {
    "accept": "Accepted links",
    "score_reject": "Lexical refusals",
    "temporal_reject": "Temporal refusals",
}

## Helpers

Pure functions. They read CONFIG names; change behaviour in CONFIG, not here.

In [3]:
def wilson(k: int, n: int, z: float = Z) -> dic<path>    """Wilson score interval — behaves sanely at the 0/n and n/n boundaries."""
    if n == 0:
        return {"k": 0, "n": 0, "p": float("nan"), "lo": float("nan"), "hi": float("nan")}
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return {"k": int(k), "n": int(n), "p": p,
            "lo": max(0.0, centre - half), "hi": min(1.0, centre + half)}


def rate(sub: pd.DataFrame, good: tuple[str, ...]) -> dic<path>    """Share of determinate labels in `good`. `indeterminate` leaves the denominator."""
    det = sub[sub["label"] != "indeterminate"]
    out = wilson(det["label"].isin(good).sum(), len(det))
    out["indeterminate"] = int((sub["label"] == "indeterminate").sum())
    return out


def fmt(r: dict) -> st<path>    if not r["n"]:
        return "n/a"
    return f"{r['k']}/{r['n']} = {r['p']:.1%}  [{r['lo']:.1%}, {r['hi']:.1%}]"


def refresh_freeze() -> Non<path>    """Re-export live labels from the review database into the freeze file."""
    app = EXPERIMENTS_DIR / "review_app"
    print(subprocess.run([sys.executable, "export_identity_labels.py"], cwd=app,
                         capture_output=True, text=True, check=True).stdout.strip())


def load() -> pd.DataFram<path>    df = pd.read_json(FREEZE, lines=True)
    df["reviewed_at"] = pd.to_datetime(df["reviewed_at"])
    return df


def prefix_length(sorted_orders: list[int]) -> in<path>    """How many items form an unbroken run from the reviewer's first item."""
    n = 1
    while n < len(sorted_orders) and sorted_orders[n] == sorted_orders[n - 1] + 1:
        n += 1
    return n

## 1 · Load, and check the audit is analysable

Three things have to hold before any rate below means anything.

**Contiguity.** The queue was shuffled once with a fixed seed, so any *unbroken prefix*
of a reviewer's block is a random subset of the population. A reviewer who skipped
around would have chosen their own sample, and the prefix argument would collapse.

**Block containment.** Nobody labelled outside their assigned range, so blocks do not
overlap and no item is double-counted.

**Blinding.** The reviewer-facing query must not expose the machine's outcome. This is
checked against the running application's own column list, not against a copy.

In [4]:
if REFRESH_FROM_D<path>    refresh_freeze()

df = load()
lab = df[df["label"].notna()].copy()
print(f"population {len(df)} items · labelled {len(lab)} ({len(lab)/len(df):.1%}) "
      f"· reviewers {lab['reviewer'].nunique()}")

rows = []
for who, g in lab.groupby("reviewer"):
    orders = sorted(g["sort_order"])
    lo, hi = BLOCKS[who]
    rows.append({
        "reviewer": who, "n": len(orders),
        "block": f"{lo}-{hi}", "labelled": f"{orders[0]}-{orders[-1]}",
        "prefix": prefix_length(orders),
        "contiguous": prefix_length(orders) == len(orders),
        "in block": orders[0] >= lo and orders[-1] <= hi,
    })
integrity = pd.DataFrame(rows).sort_values("block")
display(integrity)

assert integrity["contiguous"].all(), "a reviewer skipped items; prefix argument invalid"
assert integrity["in block"].all(), "a reviewer labelled outside their assigned block"
assert lab["id"].is_unique, "an item carries more than one label"
print("contiguity, block containment and uniqueness: OK")

population 917 items · labelled 406 (44.3%) · reviewers 5


,reviewer,n,block,labelled,prefix,contiguous,in block
0,R1,45,1-100,1-45,45,True,True
2,R2,100,101-200,101-200,100,True,True
3,R3,61,201-300,201-261,61,True,True
1,R4,100,301-400,301-400,100,True,True
4,R5,100,401-500,401-500,100,True,True


contiguity, block containment and uniqueness: OK


In [5]:
# Blinding is a property of the code, so assert against the app's own column list.
sys.path.insert(0, str(EXPERIMENTS_DIR / "review_app"))
import identity  # noqa: E402

HIDDEN = ["gate_outcome", "candidate_score", "score_threshold", "score_state",
          "temporal_state", "decision_reason", "authority_source",
          "candidate_birth_year", "candidate_death_year"]
leaked = [c for c in HIDDEN if c in identity.VISIBLE_COLS]
print("columns withheld from reviewers:", HIDDEN)
assert not leaked, f"reviewer-facing query exposes {leaked}"
print("blinding: OK — the outcome columns analysed below were never shown")

columns withheld from reviewers: ['gate_outcome', 'candidate_score', 'score_threshold', 'score_state', 'temporal_state', 'decision_reason', 'authority_source', 'candidate_birth_year', 'candidate_death_year']
blinding: OK — the outcome columns analysed below were never shown


## 2 · Coverage, and whether stopping was selective

Reviewers stopped when their session ran out, not at a planned n. That is only harmless
if stopping was **non-selective** — if people did not skim ahead and quit on the hard
items. Nothing forces that, so it has to be checked rather than assumed.

If stopping is non-selective, the labelled union should look like the population on
variables the reviewer never saw. Two such variables are the gate outcome and the
candidate score, both hidden by the blinding check above.

In [6]:
cov = pd.DataFrame({
    "population": df["gate_outcome"].value_counts(),
    "labelled": lab["gate_outcome"].value_counts(),
})
cov["coverage"] = (cov["labelled"] / cov["population"]).map("{:.1%}".format)
cov["pop share"] = (cov["population"] / len(df)).map("{:.1%}".format)
cov["lab share"] = (cov["labelled"] / len(lab)).map("{:.1%}".format)
display(cov)

ent = pd.DataFrame({
    "pop share": df["entity_type"].value_counts(normalize=True),
    "lab share": lab["entity_type"].value_counts(normalize=True),
}).map("{:.1%}".format)
display(ent)

,population,labelled,coverage,pop share,lab share
gate_outcome,,,,,
accept,561,253,45.1%,61.2%,62.3%
score_reject,319,135,42.3%,34.8%,33.3%
temporal_reject,37,18,48.6%,4.0%,4.4%


,pop share,lab share
entity_type,,
person,49.2%,50.5%
org,25.8%,25.6%
place,22.9%,21.7%
subject,2.1%,2.2%


In [7]:
# Candidate score is hidden from reviewers, so labelled and unlabelled accepts should
# not differ. A gap here would say people quit on the items that looked hard.
acc_all = df[df["gate_outcome"] == "accept"]
by_seen = acc_all.groupby(acc_all["label"].notna())["candidate_score"]
comp = by_seen.agg(["count", "mean", "median", "std"]).round(3)
comp.index = ["not yet labelled", "labelled"]
display(comp)

from scipy.stats import ks_2samp  # noqa: E402
ks = ks_2samp(acc_all.loc[acc_all["label"].notna(), "candidate_score"],
              acc_all.loc[acc_all["label"].isna(), "candidate_score"])
print(f"KS on hidden candidate score: D={ks.statistic:.3f}, p={ks.pvalue:.3f}")
print("A large p means labelled and unlabelled accepts are indistinguishable on a")
print("variable the reviewer could not see, i.e. no evidence of selective stopping.")

,count,mean,median,std
not yet labelled,308,0.902,0.930,0.106
labelled,253,0.898,0.925,0.108


KS on hidden candidate score: D=0.036, p=0.991
A large p means labelled and unlabelled accepts are indistinguishable on a
variable the reviewer could not see, i.e. no evidence of selective stopping.


## 3 · The two headline rates

**Accepted-link precision** — of the links the gate let through into records, how many
denote the right entity. This is the number that decides whether the gate's output can
be published unreviewed.

**Refusal correctness** — of the links the gate blocked, how many deserved blocking.
Reported separately for the lexical and the temporal rule, because they are different
checks and E1 already showed they are not redundant.

In [8]:
acc = lab[lab["gate_outcome"] == "accept"]
lex = lab[lab["gate_outcome"] == "score_reject"]
tmp = lab[lab["gate_outcome"] == "temporal_reject"]

precision = rate(acc, ("correct",))
lex_ok = rate(lex, GATE_ERROR_LABELS)
tmp_ok = rate(tmp, GATE_ERROR_LABELS)

headline = pd.DataFrame([
    {"metric": "Accepted-link precision", "reviewed": len(acc), **precision},
    {"metric": "Lexical-refusal correctness", "reviewed": len(lex), **lex_ok},
    {"metric": "Temporal-refusal correctness", "reviewed": len(tmp), **tmp_ok},
])
headline["estimate"] = headline.apply(
    lambda r: f"{r['p']:.1%}  [{r['lo']:.1%}, {r['hi']:.1%}]", axis=1)
display(headline[["metric", "reviewed", "indeterminate", "k", "n", "estimate"]])

print(f"\nAccepted-link precision  {fmt(precision)}")
print(f"Lexical-refusal correctness  {fmt(lex_ok)}")
print(f"Temporal-refusal correctness  {fmt(tmp_ok)}")

,metric,reviewed,indeterminate,k,n,estimate
0,Accepted-link precision,253,13,81,240,"33.8% [28.1%, 39.9%]"
1,Lexical-refusal correctness,135,2,111,133,"83.5% [76.2%, 88.8%]"
2,Temporal-refusal correctness,18,0,18,18,"100.0% [82.4%, 100.0%]"



Accepted-link precision  81/240 = 33.8%  [28.1%, 39.9%]
Lexical-refusal correctness  111/133 = 83.5%  [76.2%, 88.8%]
Temporal-refusal correctness  18/18 = 100.0%  [82.4%, 100.0%]


In [9]:
# The full contingency table the rates are drawn from.
cross = pd.crosstab(lab["gate_outcome"], lab["label"], margins=True)
cross = cross.reindex(columns=[c for c in LABEL_ORDER if c in cross.columns] + ["All"])
display(cross)

label,correct,wrong,true_nil,indeterminate,All
gate_outcome,,,,,
accept,81,130,29,13,253
score_reject,22,96,15,2,135
temporal_reject,0,16,2,0,18
All,103,242,46,15,406


### The unit matters, and it is not a rounding difference

The precision above is **per unique decision**. A decision the gate made once but reused
across twenty issues counts once, which is right for auditing the gate.

A reader of a record meets links, not decisions, so the natural question there is differen<path>of the authority links as they actually appear, how many are correct? That is the
occurrence-weighted rate, and it is **not** the same number, because reuse correlates with
correctness — correct decisions are repeated more often than wrong ones.

Both are reported. Neither is converted into the other, and neither is ever described as "the
share of authority links in the records that are wrong." An earlier version of this project
retired a headline number for exactly this class of unit slippage.

In [10]:
det_acc = acc[acc["label"] != "indeterminate"]
reuse = det_acc.groupby("label")["card_uri_occurrences"].agg(["count", "sum", "mean"]).round(2)
reuse.columns = ["decisions", "card occurrences", "mean occurrences"]
display(reuse)

w_correct = det_acc.loc[det_acc["label"] == "correct", "card_uri_occurrences"].sum()
w_total = det_acc["card_uri_occurrences"].sum()
precision_weighted = wilson(int(w_correct), int(w_total))

print(f"per unique decision      {fmt(precision)}")
print(f"per card occurrence      {fmt(precision_weighted)}")
print(f"gap                      {precision_weighted['p'] - precision['p']:+.1%}")
print("\nThe occurrence-weighted interval is optimistic: occurrences of one decision are")
print("not independent observations, so its true interval is wider than shown.")

,decisions,card occurrences,mean occurrences
label,,,
correct,81,865,10.68
true_nil,29,49,1.69
wrong,130,562,4.32


per unique decision      81/240 = 33.8%  [28.1%, 39.9%]
per card occurrence      865/1476 = 58.6%  [56.1%, 61.1%]
gap                      +24.9%

The occurrence-weighted interval is optimistic: occurrences of one decision are
not independent observations, so its true interval is wider than shown.


## 4 · Does the result survive the reviewers disagreeing?

The five reviewers do not use the codebook identically, and the disagreement is
concentrated in exactly one place: whether a bad link is called `wrong` (this URI is the
wrong entity) or `true_nil` (no suitable record exists at all).

That boundary is genuinely hard, and the table below shows reviewers land on opposite
sides of it. The important point is that **it does not move either headline rate**, because
on an accepted row `wrong` and `true_nil` are both gate errors, and on a refused row they
are both correct refusals. The disagreement is about *why* the gate was wrong, not
*whether* it was.

In [11]:
per = []
for who, g in acc.groupby("reviewer"):
    det = g[g["label"] != "indeterminate"]
    r = wilson((det["label"] == "correct").sum(), len(det))
    per.append({
        "reviewer": who, "block": f"{BLOCKS[who][0]}-{BLOCKS[who][1]}",
        "accepts reviewed": len(g),
        "precision": f"{r['p']:.1%}", "95% CI": f"[{r['lo']:.1%}, {r['hi']:.1%}]",
        "calls errors 'true_nil'": f"{(det['label'] == 'true_nil').mean():.0%}",
    })
per = pd.DataFrame(per).sort_values("block")
display(per)
print("Precision is stable across reviewers even though their use of 'true_nil'")
print("ranges from never to more than a third of all errors.")

,reviewer,block,accepts reviewed,precision,95% CI,calls errors 'true_nil'
0,R1,1-100,24,25.0%,"[12.0%, 44.9%]",4%
2,R2,101-200,54,33.3%,"[22.2%, 46.6%]",0%
3,R3,201-300,37,36.7%,"[21.9%, 54.5%]",7%
1,R4,301-400,69,29.0%,"[19.6%, 40.6%]",38%
4,R5,401-500,69,41.3%,"[30.0%, 53.6%]",0%


Precision is stable across reviewers even though their use of 'true_nil'
ranges from never to more than a third of all errors.


In [12]:
# Leave-one-reviewer-out: no single reviewer or block is driving the result.
loo = []
for who in sorted(acc["reviewer"].unique()):
    r = rate(acc[acc["reviewer"] != who], ("correct",))
    loo.append({"dropped": who, "n": r["n"], "precision": f"{r['p']:.1%}",
                "95% CI": f"[{r['lo']:.1%}, {r['hi']:.1%}]"})
loo.append({"dropped": "— none —", "n": precision["n"],
            "precision": f"{precision['p']:.1%}",
            "95% CI": f"[{precision['lo']:.1%}, {precision['hi']:.1%}]"})
display(pd.DataFrame(loo))

,dropped,n,precision,95% CI
0,R1,216,34.7%,"[28.7%, 41.3%]"
1,R4,171,35.7%,"[28.9%, 43.1%]"
2,R2,186,33.9%,"[27.5%, 40.9%]"
3,R3,210,33.3%,"[27.3%, 40.0%]"
4,R5,177,31.1%,"[24.7%, 38.2%]"
5,— none —,240,33.8%,"[28.1%, 39.9%]"


In [13]:
# Worst case for the claim: count every 'indeterminate' as if the gate were right.
opt = wilson((acc["label"].isin(["correct", "indeterminate"])).sum(), len(acc))
pes = wilson((acc["label"] == "correct").sum(), len(acc))
print(f"indeterminate excluded (reported)   {fmt(precision)}")
print(f"indeterminate counted as correct    {fmt(opt)}   <- most generous to the gate")
print(f"indeterminate counted as wrong      {fmt(pes)}")
print("\nThe bound that is most generous to the gate still leaves accepted-link")
print("precision far below anything publishable without review.")

indeterminate excluded (reported)   81/240 = 33.8%  [28.1%, 39.9%]
indeterminate counted as correct    94/253 = 37.2%  [31.4%, 43.3%]   <- most generous to the gate
indeterminate counted as wrong      81/253 = 32.0%  [26.6%, 38.0%]

The bound that is most generous to the gate still leaves accepted-link
precision far below anything publishable without review.


## 5 · How the gate fails

Severity and residual reason were collected only to characterise errors, never to compute
the rates above. They answer the question a reader asks next: is this a cosmetic
mismatch, or the wrong person?

In [14]:
wrong = lab[lab["label"] == "wrong"]
sev = wrong["severity"].value_counts().reindex(["critical", "major", "minor"]).fillna(0).astype(int)
sev_df = sev.to_frame("count")
sev_df["share"] = (sev / sev.sum()).map("{:.1%}".format)
display(sev_df)

reason = pd.crosstab(lab["residual_reason"], lab["gate_outcome"])
reason["total"] = reason.sum(axis=1)
display(reason.sort_values("total", ascending=False))

,count,share
severity,,
critical,209,86.4%
major,33,13.6%
minor,0,0.0%


gate_outcome,accept,score_reject,temporal_reject,total
residual_reason,,,,
same_name_same_era,37,19,4,60
out_of_kb,30,17,2,49
org_alias_or_parent,6,8,0,14
insufficient_context,7,7,0,14
equivalent_identifier,4,1,3,8
editorial_not_identity,3,2,0,5


In [15]:
# The dominant failure modes, stated as shares of explained wrong-or-NIL judgments.
wrong_or_nil = lab[
    lab["label"].isin(GATE_ERROR_LABELS) & lab["residual_reason"].notna()
]
top = wrong_or_nil["residual_reason"].value_counts()
for name, n in top.items():
    print(f"{name:24s} {n:4d}  {n / top.sum():.1%}")
all_wrong_or_nil = int(lab["label"].isin(GATE_ERROR_LABELS).sum())
print(f"\n{len(wrong_or_nil)} of {all_wrong_or_nil} wrong-or-NIL judgments carry an explanation.")
print("\nsame_name_same_era is the failure a lexical score cannot fix by tuning: the")
print("strings genuinely match and the lifespans genuinely overlap. out_of_kb is a")
print("different problem — the right record does not exist to be found.")

same_name_same_era         59  41.8%
out_of_kb                  49  34.8%
org_alias_or_parent        14  9.9%
equivalent_identifier       8  5.7%
insufficient_context        6  4.3%
editorial_not_identity      5  3.5%

141 of 288 wrong-or-NIL judgments carry an explanation.

same_name_same_era is the failure a lexical score cannot fix by tuning: the
strings genuinely match and the lifespans genuinely overlap. out_of_kb is a
different problem — the right record does not exist to be found.


## 6 · What the audit costs

Two clocks disagree, and the difference is itself a finding.

`active_ms` counts only time with the review tab focused. Reviewers spend much of an item
in *another* tab reading the authority record, so this timer systematically undercounts.
Wall-clock between consecutive submissions captures that, minus long breaks.

Report the wall-clock figure. The focused timer is kept here to show the size of the gap.

In [16]:
eff = []
for who, g in lab.sort_values("reviewed_at").groupby("reviewer"):
    gaps = g["reviewed_at"].diff().dt.total_seconds().dropna()
    working = gaps[gaps <= BREAK_THRESHOLD_S].sum() / 60
    focused = g["active_ms"].fillna(0).sum() / 60000
    eff.append({
        "reviewer": who, "items": len(g),
        "wall-clock min": round(working, 1),
        "min/item": round(working / len(g), 2),
        "focused min": round(focused, 1),
        "focused share": f"{focused / working:.0%}" if working else "n/a",
        "breaks": int((gaps > BREAK_THRESHOLD_S).sum()),
    })
eff = pd.DataFrame(eff).sort_values("min/item")
display(eff)

per_item = eff["wall-clock min"].sum() / eff["items"].sum()
print(f"\npooled {per_item:.2f} min/item wall-clock vs "
      f"{lab['active_ms'].fillna(0).sum()/60000/len(lab):.2f} min/item focused")
print(f"remaining {len(df)-len(lab)} items -> about "
      f"{(len(df)-len(lab))*per_item/60:.1f} more person-hours")
print(f"full population of {len(df)} -> about {len(df)*per_item/60:.1f} person-hours")

,reviewer,items,wall-clock min,min/item,focused min,focused share,breaks
2,R2,100,127.3,1.27,46.1,36%,0
3,R3,61,78.4,1.29,48.2,61%,4
4,R5,100,139.3,1.39,69.5,50%,10
1,R4,100,144.6,1.45,47.7,33%,4
0,R1,45,66.5,1.48,49.3,74%,12



pooled 1.37 min/item wall-clock vs 0.64 min/item focused
remaining 511 items -> about 11.7 more person-hours
full population of 917 -> about 20.9 person-hours


## 7 · Figure preview

Same shape as the E3 figure in the manuscript: stacked outcome counts, with the headline
rate as a labelled column on the right. This is a **preview only** — the manuscript
version is hand-written TikZ, built once the audit is frozen.

Read the colours carefully: bars are coloured by the **human label**, and the label means
opposite things for the two rows. On the accepted row `correct` (blue) is the gate being
right; on the refusal rows `correct` (blue) is the gate being *wrong*, because it blocked
a URI that was actually the right one. That inversion is inherent to blinded labelling,
and the manuscript figure has to resolve it — either by colouring on gate correctness
instead of on the label, or by splitting the two panels.

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COLOURS = {"correct": "#486E9E", "wrong": "#E8A860",
           "true_nil": "#C8C8C8", "indeterminate": "#EDEDED"}

order = ["accept", "score_reject", "temporal_reject"]
rates = {"accept": precision, "score_reject": lex_ok, "temporal_reject": tmp_ok}
right = {"accept": "precision", "score_reject": "refusal ok", "temporal_reject": "refusal ok"}

fig, ax = plt.subplots(figsize=(9, 2.6))
for y, outcome in enumerate(order):
    sub = lab[lab["gate_outcome"] == outcome]
    left = 0
    for lbl in LABEL_ORDE<path>        n = int((sub["label"] == lbl).sum())
        if not <path>            continue
        ax.barh(y, n, left=left, color=COLOURS[lbl], edgecolor="black", linewidth=0.4)
        if n >= 6:
            ax.text(left + n / 2, y, str(n), ha="center", va="center", fontsize=9,
                    color="white" if lbl == "correct" else "black")
        left += n
    r = rates[outcome]
    ax.text(len(lab[lab["gate_outcome"] == "accept"]) * 1.06, y,
            f"{right[outcome]} {r['p']:.0%}", va="center", fontsize=9)

ax.set_yticks(range(len(order)))
ax.set_yticklabels([OUTCOME_LABEL[o] for o in order], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel("unique identity decisions reviewed", fontsize=9)
ax.set_xlim(0, len(lab[lab["gate_outcome"] == "accept"]) * 1.3)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=COLOURS[l], ec="black", lw=0.4)
                   for l in LABEL_ORDER],
          labels=["correct", "wrong", "true NIL", "indeterminate"],
          ncol=4, frameon=False, loc="lower center", bbox_to_anchor=(0.42, 1.02), fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "e2_identity_audit_preview.png", dpi=200, bbox_inches="tight")
plt.show()

## 8 · Export

Counts and rates only — no per-item labels, so the artifact can be committed. The
manuscript reads its numbers from this file.

In [18]:
explained_wrong_or_nil = lab[
    lab["label"].isin(GATE_ERROR_LABELS) & lab["residual_reason"].notna()
]

summary = {
    "generated_from": FREEZE.name,
    "status": "SUBMISSION FREEZE — 2026-09-05 21:30 UTC; fixed 406-decision analysis cutoff",
    "population": int(len(df)),
    "labelled": int(len(lab)),
    "coverage": round(len(lab) / len(df), 4),
    "reviewers": int(lab["reviewer"].nunique()),
    "blocks": {r["reviewer"]: {"assigned": r["block"], "completed_prefix": int(r["prefix"])}
               for r in integrity.to_dict("records")},
    "items_by_outcome": df["gate_outcome"].value_counts().to_dict(),
    "outcome_x_label": {f"{o}|{l}": int(n)
                        for (o, l), n in lab.groupby(["gate_outcome", "label"]).size().items()},
    "rates": {
        "accepted_link_precision": precision,
        "accepted_link_precision_unit": "unique (entity_type, surface, candidate_uri) decision",
        "accepted_link_precision_occurrence_weighted": precision_weighted,
        "occurrence_weighted_note": "weights each decision by card_uri_occurrences; its "
                                    "interval is optimistic because occurrences of one "
                                    "decision are not independent",
        "lexical_refusal_correctness": lex_ok,
        "temporal_refusal_correctness": tmp_ok,
    },
    "severity_of_wrong": sev.to_dict(),
    "residual_reason_all_labels": {
        "definition": "all reviewed rows carrying a residual reason",
        "n": int(lab["residual_reason"].notna().sum()),
        "counts": lab["residual_reason"].value_counts().to_dict(),
    },
    "residual_reason_wrong_or_true_nil": {
        "definition": "manuscript subset: rows labeled wrong or true_nil and carrying a residual reason",
        "n": int(len(explained_wrong_or_nil)),
        "counts": explained_wrong_or_nil["residual_reason"].value_counts().to_dict(),
    },
    "effort": {
        "wall_clock_min_per_item": round(per_item, 3),
        "focused_min_per_item": round(lab["active_ms"].fillna(0).sum() / 60000 / len(lab), 3),
        "projected_person_hours_full_population": round(len(df) * per_item / 60, 1),
    },
}
ARTIFACT.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"wrote {ARTIFACT}")
print(json.dumps(summary["rates"], indent=2))

wrote <path>
{
  "accepted_link_precision": {
    "k": 81,
    "n": 240,
    "p": 0.3375,
    "lo": 0.28065337442469745,
    "hi": 0.3994668347602816,
    "indeterminate": 13
  },
  "accepted_link_precision_unit": "unique (entity_type, surface, candidate_uri) decision",
  "accepted_link_precision_occurrence_weighted": {
    "k": 865,
    "n": 1476,
    "p": 0.5860433604336044,
    "lo": 0.5607237963114313,
    "hi": 0.6109161954282252
  },
  "occurrence_weighted_note": "weights each decision by card_uri_occurrences; its interval is optimistic because occurrences of one decision are not independent",
  "lexical_refusal_correctness": {
    "k": 111,
    "n": 133,
    "p": 0.8345864661654135,
    "lo": 0.7622347543029068,
    "hi": 0.8881522625106938,
    "indeterminate": 2
  },
  "temporal_refusal_correctness": {
    "k": 18,
    "n": 18,
    "p": 1.0,
    "lo": 0.8241154494176252,
    "hi": 1.0,
    "indeterminate": 0
  }
}


## What this establishes, and what it does not

The gate is **strongly asymmetric**. Its refusals are largely justified, but the links it
lets through are wrong far more often than not. Traceability and correctness are
therefore separate properties: E1's complete, inspectable decision path says nothing
about whether the decisions were right, which is precisely the claim separation the paper
argues for.

The dominant failure, `same_name_same_era`, is not a threshold-tuning problem. The
candidate's string matches and its lifespan overlaps, so no setting of a lexical score
distinguishes it. Resolving it needs evidence the gate never consults — the surrounding
article text — which is where an LLM would plausibly help, and is future work rather than
a result here.

Three limits on the numbers above. Coverage is partial: this submission freeze estimates
rates over completed prefixes. The blocks are contiguous prefixes of a seeded shuffle rather
than one simple random sample, so §2's checks carry the representativeness argument.
And the reviewers are the authors and their students, not independent catalogers.